In [ ]:
# ============================
# Full pipeline: single cell
# ============================
# NOTE: If required, run (once) in Colab:
!pip install -q scikit-learn xgboost shap lime imbalanced-learn openpyxl xlrd

import os, time, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from packaging import version
import sklearn

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.base import clone
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

import shap
from lime.lime_tabular import LimeTabularExplainer

# ---------------------
# Config
# ---------------------
RANDOM_STATE = 42
TEST_SIZE = 0.30
TARGET_N = 5269
ARTIFACTS = "artifacts"
os.makedirs(ARTIFACTS, exist_ok=True)
np.random.seed(RANDOM_STATE)
plt.rcParams['figure.figsize'] = (8,5)

# ---------------------
# FAST / FULL switch
# ---------------------
FAST_MODE = False   # set False for full experiments (slower)
if FAST_MODE:
    print("=== FAST MODE (quick debugging) ===")
else:
    print("=== FULL MODE (full experiments) ===")

# ---------------------
# Helper functions
# ---------------------
def evaluate_all(y_true, y_pred, y_score=None):
    out = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0)
    }
    try:
        if y_score is not None:
            out["roc_auc_ovr"] = roc_auc_score(pd.get_dummies(y_true), y_score, average="macro", multi_class="ovr")
        else:
            out["roc_auc_ovr"] = np.nan
    except Exception:
        out["roc_auc_ovr"] = np.nan
    return out

def binarize_series(s: pd.Series):
    """Numeric -> median split; Categorical -> mode vs rest"""
    try:
        if pd.api.types.is_numeric_dtype(s):
            arr = pd.to_numeric(s, errors="coerce")
            thr = np.nanmedian(arr)
            yb = (arr > thr).astype(int).fillna(0).astype(int)
            return yb.values, {"rule": "numeric_median", "threshold": float(thr)}
        else:
            sm = s.astype(str)
            mode_lbl = sm.mode().iloc[0]
            yb = (sm != mode_lbl).astype(int)
            return yb.values, {"rule": "categorical_mode_vs_rest", "negative_class": mode_lbl}
    except Exception as e:
        sm = s.astype(str)
        mode_lbl = sm.mode().iloc[0]
        yb = (sm != mode_lbl).astype(int)
        return yb.values, {"rule": "fallback_mode_vs_rest", "negative_class": mode_lbl}

# ---------------------
# Load dataset (CSV)
# ---------------------
DATA_FILE = "processed_data_1.csv"   # change if needed
if not os.path.exists(DATA_FILE):
    raise FileNotFoundError(f"File not found: {DATA_FILE} — upload it or set DATA_FILE correctly.")

df = pd.read_csv(DATA_FILE, encoding='latin-1')  # try latin-1 fallback
print("Raw shape:", df.shape)
display(df.head())

# ---------------------
# Drop obvious ID-like columns
# ---------------------
drop_like = ["id","user_id","email","name","timestamp"]
for d in drop_like:
    if d in df.columns:
        df.drop(columns=[d], inplace=True)
print("After dropping ids:", df.shape)

# ---------------------
# Detect / set target columns
# ---------------------
# Try to auto-detect; if wrong, edit the mapping below manually.
TARGETS = {}
for c in df.columns:
    lc = c.lower()
    if ('depress' in lc or 'phq' in lc) and 'depression' not in TARGETS:
        TARGETS['depression'] = c
    if ('satisf' in lc or 'swls' in lc or 'life_satisfaction' in lc or ('life' in lc and 'satisfaction' in lc)) and 'life_satisfaction' not in TARGETS:
        TARGETS['life_satisfaction'] = c
    if ('person' in lc or 'bfi' in lc or 'bigfive' in lc or 'big five' in lc) and 'personality' not in TARGETS:
        TARGETS['personality'] = c

print("Auto-detected TARGETS:", TARGETS)
if len(TARGETS) < 1:
    raise RuntimeError("No targets auto-detected — set TARGETS manually to your target column names.")
# If auto-detection is incorrect, you can overwrite TARGETS here, e.g.:
# TARGETS = {'depression':'depression_level', 'life_satisfaction':'swls', 'personality':'bfi_total'}

target_cols = list(TARGETS.values())
feat_df = df.drop(columns=[c for c in target_cols if c in df.columns], errors="ignore")
Y_raw = df[target_cols].copy()

# ---------------------
# Preprocessing (impute before augmentation)
# ---------------------
num_cols = feat_df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in feat_df.columns if c not in num_cols]

feat_filled = feat_df.copy()
if num_cols:
    num_med = feat_df[num_cols].median()
    feat_filled[num_cols] = feat_df[num_cols].fillna(num_med)
if cat_cols:
    for c in cat_cols:
        mode_val = feat_df[c].mode().iloc[0] if feat_df[c].notna().any() else "NA"
        feat_filled[c] = feat_df[c].fillna(mode_val)

# ---------------------
# Augmentation -> exactly TARGET_N rows
# ---------------------
n_current = feat_filled.shape[0]
print("Original rows:", n_current)
if n_current < TARGET_N:
    need = TARGET_N - n_current
    rng = np.random.default_rng(RANDOM_STATE)
    idx = rng.integers(0, n_current, size=need)
    feat_syn = feat_filled.iloc[idx].copy()
    if num_cols:
        stds = feat_filled[num_cols].std().replace(0,1e-12)
        noise = pd.DataFrame(rng.normal(0,1,size=(need,len(num_cols))), columns=num_cols)
        feat_syn[num_cols] = feat_syn[num_cols] + 0.10 * noise.values * stds.values
    Y_syn = Y_raw.iloc[idx].reset_index(drop=True)
    feat_aug = pd.concat([feat_filled, feat_syn], ignore_index=True)
    Y_aug = pd.concat([Y_raw, Y_syn], ignore_index=True)
else:
    feat_aug = feat_filled.copy()
    Y_aug = Y_raw.copy()

print("After augmentation rows:", feat_aug.shape[0])

# ---------------------
# Vectorize features (Version-safe OneHotEncoder)
# ---------------------
# OneHotEncoder sparse parameter name changed between sklearn versions
if version.parse(sklearn.__version__) >= version.parse("1.2"):
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
else:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

numeric_pipe = Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())])
categorical_pipe = Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", ohe)])

ct = ColumnTransformer(transformers=[("num", numeric_pipe, num_cols),
                                     ("cat", categorical_pipe, cat_cols)],
                       remainder="drop")

print("Fitting transformer... (this may take a few seconds)")
X_all = ct.fit_transform(feat_aug)                      # numpy array
feature_names = ct.get_feature_names_out().tolist()
print("Transformed shape:", X_all.shape, "| n_features:", len(feature_names))

# ---------------------
# Prepare model configs (FAST vs FULL)
# ---------------------
if FAST_MODE:
    BASE_MODELS = {
        "LogisticRegression": LogisticRegression(max_iter=500, random_state=RANDOM_STATE),
        "XGBoost": XGBClassifier(objective="binary:logistic", eval_metric="logloss",
                                 n_estimators=20, max_depth=3, learning_rate=0.1, random_state=RANDOM_STATE, n_jobs=2)
    }
    STACK_CV = 2
    RUN_HYBRID = False
else:
    BASE_MODELS = {
        "LogisticRegression": LogisticRegression(max_iter=800, random_state=RANDOM_STATE),
        "DecisionTree": DecisionTreeClassifier(random_state=RANDOM_STATE),
        "SVM": SVC(probability=True, random_state=RANDOM_STATE),
        "GaussianNB": GaussianNB(),
        "XGBoost": XGBClassifier(objective="binary:logistic", eval_metric="logloss",
                                 n_estimators=100, max_depth=4, learning_rate=0.1, random_state=RANDOM_STATE, n_jobs=2)
    }
    STACK_CV = 3
    RUN_HYBRID = True

# ---------------------
# Training loop (per target, binary)
# ---------------------
trained = {}
rows = []

for tkey, tcol in TARGETS.items():
    print("\n" + "="*60)
    print("TARGET:", tkey, "| column:", tcol)
    y_series = Y_aug[tcol]
    y_bin, rule = binarize_series(y_series)
    print("Binarize rule:", rule)

    # train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X_all, y_bin, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_bin)
    print("Train/test shapes:", X_train.shape, X_test.shape)

    # SMOTE on train
    sm = SMOTE(random_state=RANDOM_STATE)
    X_train_res, y_train_res = sm.fit_resample(X_train, y_train)
    print("After SMOTE class counts:", np.bincount(y_train_res))

    fitted = {}
    # base models
    for name, est in BASE_MODELS.items():
        try:
            m = clone(est)
            t0 = time.time()
            m.fit(X_train_res, y_train_res)
            dur = time.time() - t0
            y_pred = m.predict(X_test)
            y_proba = m.predict_proba(X_test)[:,1] if hasattr(m, "predict_proba") else None
            metrics = evaluate_all(y_test, y_pred, (np.c_[1-y_proba, y_proba] if y_proba is not None else None))
            rows.append({"target": tkey, "model": name, "train_time_s": round(dur,2), **metrics})
            fitted[name] = m
            print(f"  {name}: acc={metrics['accuracy']:.3f} f1={metrics['f1_macro']:.3f} time={dur:.1f}s")
        except Exception as e:
            print(f"  {name} failed:", e)

    # Stack1: fresh estimators
    try:
        xgb_b = XGBClassifier(objective="binary:logistic", eval_metric="logloss",
                              n_estimators=(20 if FAST_MODE else 100),
                              max_depth=(3 if FAST_MODE else 4),
                              random_state=RANDOM_STATE, n_jobs=2)
        rf_b = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=2)
        stack1 = StackingClassifier(
            estimators=[("xgb", xgb_b), ("rf", rf_b)],
            final_estimator=LogisticRegression(max_iter=800),
            cv=STACK_CV, n_jobs=-1
        )
        t0=time.time(); stack1.fit(X_train_res, y_train_res); dur=time.time()-t0
        yp = stack1.predict(X_test); ypb = stack1.predict_proba(X_test) if hasattr(stack1,"predict_proba") else None
        met = evaluate_all(y_test, yp, ypb)
        rows.append({"target": tkey, "model": "Stack1", "train_time_s": round(dur,2), **met})
        fitted["Stack1"] = stack1
        print(f"  Stack1: acc={met['accuracy']:.3f} f1={met['f1_macro']:.3f} time={dur:.1f}s")
    except Exception as e:
        print("  Stack1 failed:", e)

    # Stack2
    try:
        stack2 = StackingClassifier(
            estimators=[("lr", LogisticRegression(max_iter=800, random_state=RANDOM_STATE)),
                        ("gb", GradientBoostingClassifier(random_state=RANDOM_STATE))],
            final_estimator=LogisticRegression(max_iter=800),
            cv=STACK_CV, n_jobs=-1
        )
        t0=time.time(); stack2.fit(X_train_res, y_train_res); dur=time.time()-t0
        yp = stack2.predict(X_test); ypb = stack2.predict_proba(X_test) if hasattr(stack2,"predict_proba") else None
        met = evaluate_all(y_test, yp, ypb)
        rows.append({"target": tkey, "model": "Stack2", "train_time_s": round(dur,2), **met})
        fitted["Stack2"] = stack2
        print(f"  Stack2: acc={met['accuracy']:.3f} f1={met['f1_macro']:.3f} time={dur:.1f}s")
    except Exception as e:
        print("  Stack2 failed:", e)

    # Hybrid (optional)
    if RUN_HYBRID:
        try:
            svm_feat = SVC(kernel="linear", probability=True, random_state=RANDOM_STATE)
            svm_feat.fit(X_train_res, y_train_res)
            try:
                df_train = svm_feat.decision_function(X_train_res)
                df_test  = svm_feat.decision_function(X_test)
            except Exception:
                df_train = svm_feat.predict_proba(X_train_res)[:,1:2]
                df_test  = svm_feat.predict_proba(X_test)[:,1:2]
            if df_train.ndim == 1: df_train = df_train.reshape(-1,1)
            if df_test.ndim == 1: df_test = df_test.reshape(-1,1)
            Xtr_h = np.hstack([X_train_res, df_train])
            Xte_h = np.hstack([X_test, df_test])
            xgb_h = XGBClassifier(objective="binary:logistic", eval_metric="logloss",
                                  n_estimators=(20 if FAST_MODE else 100),
                                  max_depth=(3 if FAST_MODE else 4),
                                  random_state=RANDOM_STATE, n_jobs=2)
            t0=time.time(); xgb_h.fit(Xtr_h, y_train_res); dur=time.time()-t0
            yp = xgb_h.predict(Xte_h); ypb = xgb_h.predict_proba(Xte_h)
            met = evaluate_all(y_test, yp, ypb)
            rows.append({"target": tkey, "model": "Hybrid", "train_time_s": round(dur,2), **met})
            fitted["Hybrid"] = (svm_feat, xgb_h)
            print(f"  Hybrid: acc={met['accuracy']:.3f} f1={met['f1_macro']:.3f} time={dur:.1f}s")
        except Exception as e:
            print("  Hybrid failed:", e)
    else:
        print("  Skipping Hybrid (FAST_MODE)")

    trained[tkey] = {"feature_names": feature_names,
                     "X_train_res": X_train_res, "y_train_res": y_train_res,
                     "X_test": X_test, "y_test": y_test,
                     "models": fitted}

    # free memory a bit
    gc.collect()

# ---------------------
# Save metrics
# ---------------------
perf = pd.DataFrame(rows)
perf.to_csv(os.path.join(ARTIFACTS, "performance_summary.csv"), index=False)
print("\nSaved metrics ->", os.path.join(ARTIFACTS, "performance_summary.csv"))
display(perf)

# ---------------------
# SHAP & LIME (saved to artifacts) - run now in same cell
# ---------------------
print("\nGenerating SHAP and LIME artifacts (XGBoost/LIME) ...")

for tkey, pack in trained.items():
    print("\n--- Explanations for target:", tkey, "---")
    X_test = pack["X_test"]
    X_train_res = pack["X_train_res"]
    feat_names = pack["feature_names"]
    models = pack["models"]

    # SHAP for XGBoost (if available)
    if "XGBoost" in models:
        try:
            xgb = models["XGBoost"]
            explainer = shap.TreeExplainer(xgb)
            shap_vals = explainer.shap_values(X_test)    # binary -> shape (n, f)
            # ensure shapes match
            min_cols = min(shap_vals.shape[1], X_test.shape[1], len(feat_names))
            shap.summary_plot(shap_vals[:, :min_cols], X_test[:, :min_cols], feature_names=feat_names[:min_cols], show=False)
            out_png = os.path.join(ARTIFACTS, f"shap_summary_{tkey}.png")
            plt.tight_layout(); plt.savefig(out_png); plt.close()
            print("  Saved SHAP summary ->", out_png)
            # force plot for first sample
            expected = explainer.expected_value
            fv = shap_vals[0, :min_cols]
            force = shap.force_plot(expected, fv, X_test[0, :min_cols], feature_names=feat_names[:min_cols])
            out_html = os.path.join(ARTIFACTS, f"shap_force_{tkey}.html")
            shap.save_html(out_html, force)
            print("  Saved SHAP force ->", out_html)
        except Exception as e:
            print("  SHAP failed:", e)
    else:
        print("  No XGBoost model for SHAP.")

    # LIME (choose probabilistic model)
    try:
        model_for_lime = None
        if "XGBoost" in models:
            model_for_lime = models["XGBoost"]
        else:
            # pick any with predict_proba
            for m in models.values():
                if hasattr(m, "predict_proba"):
                    model_for_lime = m
                    break
        if model_for_lime is None:
            print("  No probabilistic model available for LIME -> skipping")
        else:
            expl = LimeTabularExplainer(X_train_res, feature_names=feat_names, class_names=["0","1"], discretize_continuous=True)
            explanation = expl.explain_instance(X_test[0], model_for_lime.predict_proba, num_features=10)
            out_txt = os.path.join(ARTIFACTS, f"lime_{tkey}.txt")
            with open(out_txt, "w") as f:
                for feat, val in explanation.as_list():
                    f.write(f"{feat} : {val}\n")
            print("  Saved LIME ->", out_txt)
    except Exception as e:
        print("  LIME failed:", e)

print("\nAll done. Artifacts folder contains plots, HTML and performance CSV.")


In [ ]:
# =====================================
# Extra: Counterfactuals + XAI Metrics
# =====================================
!pip install -q dice-ml

import dice_ml
from sklearn.metrics import r2_score
from scipy.spatial.distance import cosine


import dice_ml
import random
import seaborn as sns

# -------------------------
# Counterfactual Explanations (DiCE)
# -------------------------
def generate_counterfactuals(model, X_train, y_train, X_test, feat_names, n_cf=3, sample_idx=0):
    """
    Generate counterfactuals using DiCE for one sample.
    """
    df_train = pd.DataFrame(X_train, columns=feat_names)
    df_train["label"] = y_train

    d = dice_ml.Data(dataframe=df_train,
                     continuous_features=feat_names,
                     outcome_name="label")

    m = dice_ml.Model(model=model, backend="sklearn")
    exp = dice_ml.Dice(d, m, method="random")

    x0 = pd.DataFrame([X_test[sample_idx]], columns=feat_names)

    try:
        cf = exp.generate_counterfactuals(x0, total_CFs=n_cf, desired_class="opposite")
        return cf.cf_examples_list[0].final_cfs_df
    except Exception as e:
        print("Counterfactual generation failed:", e)
        return None

# -------------------------
# Interpretability Metrics
# -------------------------
def fidelity_score(model, explainer, X_sample, feat_names, n_features=10):
    try:
        explanation = explainer.explain_instance(X_sample, model.predict_proba, num_features=n_features)
        weights = dict(explanation.as_list())
        approx = sum([X_sample[feat_names.index(f.split(" ")[0])] * w
                      for f, w in weights.items() if f.split(" ")[0] in feat_names])
        true_prob = model.predict_proba([X_sample])[0][1]
        return r2_score([true_prob], [approx])
    except Exception as e:
        print("Fidelity failed:", e)
        return np.nan

def stability_score(model, explainer, X_sample, feat_names, perturb_scale=0.01, n_features=10):
    try:
        exp1 = explainer.explain_instance(X_sample, model.predict_proba, num_features=n_features)
        perturbed = X_sample + np.random.normal(0, perturb_scale, size=len(X_sample))
        exp2 = explainer.explain_instance(perturbed, model.predict_proba, num_features=n_features)

        w1 = dict(exp1.as_list())
        w2 = dict(exp2.as_list())
        common = set(w1.keys()).intersection(set(w2.keys()))
        if not common:
            return np.nan
        v1 = np.array([w1[c] for c in common])
        v2 = np.array([w2[c] for c in common])
        return 1 - cosine(v1, v2)
    except Exception as e:
        print("Stability failed:", e)
        return np.nan

# -------------------------
# Heatmap of Feature Changes
# -------------------------
def plot_agg_heatmap(all_cf, feat_names, title, out_path):
    """
    Plot aggregated heatmap across all models:
    how often each feature changes in any counterfactual.
    """
    # baseline = first row features
    base = all_cf.iloc[0][feat_names]
    changes = (all_cf[feat_names] != base).astype(int)
    freq = changes.sum().sort_values(ascending=False)

    plt.figure(figsize=(10, 6))
    sns.heatmap(freq.to_frame().T, cmap="Blues", annot=True, cbar=False,
                xticklabels=freq.index, yticklabels=["Changed Count"])
    plt.title(title, fontsize=14)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(out_path)
    plt.close()
    print(f"    Saved aggregated heatmap -> {out_path}")

# -------------------------
# Run on trained targets
# -------------------------
extra_rows = []
N_SAMPLES = 5   # random test samples per model
N_CF = 3        # counterfactuals per sample

for tkey, pack in trained.items():
    print(f"\n>>> Extra XAI metrics for target: {tkey}")
    X_train = pack["X_train_res"]
    y_train = pack["y_train_res"]
    X_test  = pack["X_test"]
    feat_names = pack["feature_names"]
    models = pack["models"]

    # --- Fidelity & Stability (XGBoost only) ---
    if "XGBoost" in models:
        model = models["XGBoost"]
        lime_explainer = LimeTabularExplainer(X_train,
                                              feature_names=feat_names,
                                              class_names=["0","1"],
                                              discretize_continuous=True)

        fid = fidelity_score(model, lime_explainer, X_test[0], feat_names)
        stab = stability_score(model, lime_explainer, X_test[0], feat_names)
        print(f"  Fidelity: {fid:.3f} | Stability: {stab:.3f}")

        extra_rows.append({"target": tkey, "model": "XGBoost",
                           "fidelity": fid, "stability": stab})
    else:
        print("  Skipping: no XGBoost for fidelity/stability")

    # --- Counterfactuals (XGB, SVM, Stack1, Stack2) ---
    selected_models = {k: v for k, v in models.items() if k in ["XGBoost", "SVM", "Stack1", "Stack2"]}
    sample_indices = random.sample(range(len(X_test)), min(N_SAMPLES, len(X_test)))
    all_cf = []

    for mname, model in selected_models.items():
        print(f"  Model: {mname}")
        for idx in sample_indices:
            cf = generate_counterfactuals(model, X_train, y_train, X_test, feat_names,
                                          n_cf=N_CF, sample_idx=idx)
            if cf is not None:
                cf["sample_index"] = idx
                cf["model"] = mname
                cf["target"] = tkey
                all_cf.append(cf)

    if all_cf:
        cf_full = pd.concat(all_cf, ignore_index=True)
        out_csv = os.path.join(ARTIFACTS, f"counterfactuals_{tkey}_ALL.csv")
        cf_full.to_csv(out_csv, index=False)
        print(f"  Saved aggregated counterfactuals -> {out_csv}")

        # --- Aggregated Heatmap ---
        out_png = os.path.join(ARTIFACTS, f"cf_heatmap_{tkey}_ALL.png")
        plot_agg_heatmap(cf_full, feat_names, f"Aggregated Feature Changes in Counterfactuals\nTarget: {tkey}", out_png)

# -------------------------
# Merge with performance_summary.csv
# -------------------------
perf_file = os.path.join(ARTIFACTS, "performance_summary.csv")
if os.path.exists(perf_file):
    perf = pd.read_csv(perf_file)
    perf = perf.merge(pd.DataFrame(extra_rows), on=["target","model"], how="left")
    perf.to_csv(perf_file, index=False)
    print("Updated CSV with interpretability metrics ->", perf_file)
    display(perf)
else:
    print("performance_summary.csv not found!")

In [ ]:
# =====================================
# Normalized Percentage Heatmaps (per target)
# =====================================
import seaborn as sns

def plot_percentage_heatmap(cf_csv, feat_names, target, out_path):
    """
    Plot percentage-based heatmap of feature changes across all counterfactuals for a given target.
    """
    cf_full = pd.read_csv(cf_csv)

    # ensure only feature columns are checked
    feat_cols = [c for c in cf_full.columns if c in feat_names]

    # baseline = first row features
    base = cf_full.iloc[0][feat_cols]
    changes = (cf_full[feat_cols] != base).astype(int)

    # percentage of times each feature changed
    freq = (changes.sum() / len(changes)) * 100
    freq = freq.sort_values(ascending=False)

    plt.figure(figsize=(10, 6))
    sns.heatmap(freq.to_frame().T, cmap="Blues", annot=True, fmt=".1f", cbar=False,
                xticklabels=freq.index, yticklabels=["% Changed"])
    plt.title(f"Percentage of Feature Changes in Counterfactuals\nTarget: {target}", fontsize=14)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(out_path)
    plt.close()
    print(f"Saved percentage heatmap -> {out_path}")

# -------------------------
# Run for all targets
# -------------------------
for tkey, pack in trained.items():
    cf_csv = os.path.join(ARTIFACTS, f"counterfactuals_{tkey}_ALL.csv")
    if os.path.exists(cf_csv):
        out_png = os.path.join(ARTIFACTS, f"cf_heatmap_{tkey}_ALL_percent.png")
        plot_percentage_heatmap(cf_csv, pack["feature_names"], tkey, out_png)
    else:
        print(f"No counterfactuals file found for {tkey}")


Saved percentage heatmap -> artifacts/cf_heatmap_depression_ALL_percent.png
Saved percentage heatmap -> artifacts/cf_heatmap_life_satisfaction_ALL_percent.png
Saved percentage heatmap -> artifacts/cf_heatmap_personality_ALL_percent.png


In [ ]:
# =====================================
# Aggregated Comparison of Top Features Across Targets
# =====================================

def get_feature_change_percentages(cf_csv, feat_names):
    """Return % of feature changes from a counterfactual CSV."""
    cf_full = pd.read_csv(cf_csv)
    feat_cols = [c for c in cf_full.columns if c in feat_names]

    base = cf_full.iloc[0][feat_cols]
    changes = (cf_full[feat_cols] != base).astype(int)
    freq = (changes.sum() / len(changes)) * 100
    return freq.sort_values(ascending=False)

# Collect results
all_results = {}
for tkey, pack in trained.items():
    cf_csv = os.path.join(ARTIFACTS, f"counterfactuals_{tkey}_ALL.csv")
    if os.path.exists(cf_csv):
        freq = get_feature_change_percentages(cf_csv, pack["feature_names"])
        all_results[tkey] = freq
    else:
        print(f"No counterfactuals file for {tkey}")

# ------------------------------------
# Barplots of Top-10 Features Per Target
# ------------------------------------
for tkey, freq in all_results.items():
    top10 = freq.head(10)
    plt.figure(figsize=(8,5))
    sns.barplot(x=top10.values, y=top10.index, palette="Blues_r")
    plt.xlabel("% of Counterfactuals Changed")
    plt.title(f"Top-10 Feature Changes in Counterfactuals\nTarget: {tkey}")
    out_png = os.path.join(ARTIFACTS, f"cf_top10_{tkey}.png")
    plt.tight_layout()
    plt.savefig(out_png)
    plt.close()
    print(f"Saved barplot -> {out_png}")

# ------------------------------------
# Combined Heatmap (all targets)
# ------------------------------------
# Merge into single dataframe
df_compare = pd.DataFrame(all_results).fillna(0)

# Keep only top 10 features overall
top_features = df_compare.mean(axis=1).sort_values(ascending=False).head(10).index
df_top = df_compare.loc[top_features]

plt.figure(figsize=(10,6))
sns.heatmap(df_top, cmap="Blues", annot=True, fmt=".1f", cbar=True,
            xticklabels=df_top.columns, yticklabels=df_top.index)
plt.title("Top Features Across Targets (% Changed in Counterfactuals)", fontsize=14)
plt.xlabel("Target")
plt.ylabel("Feature")
plt.tight_layout()
out_png = os.path.join(ARTIFACTS, "cf_top_features_comparison.png")
plt.savefig(out_png)
plt.close()
print(f"Saved combined heatmap -> {out_png}")


In [ ]:
# =====================================
# Export Counterfactual Results to LaTeX Tables
# =====================================

# Ensure results are collected
if 'all_results' not in locals():
    all_results = {}
    for tkey, pack in trained.items():
        cf_csv = os.path.join(ARTIFACTS, f"counterfactuals_{tkey}_ALL.csv")
        if os.path.exists(cf_csv):
            freq = get_feature_change_percentages(cf_csv, pack["feature_names"])
            all_results[tkey] = freq

# Export top 10 per target
for tkey, freq in all_results.items():
    top10 = freq.head(10)
    df_tex = pd.DataFrame(top10).reset_index()
    df_tex.columns = ["Feature", "% Changed"]
    out_tex = os.path.join(ARTIFACTS, f"cf_top10_{tkey}.tex")
    with open(out_tex, "w") as f:
        f.write(df_tex.to_latex(index=False, float_format="%.1f"))
    print(f"Saved LaTeX table -> {out_tex}")

# Export combined heatmap table
df_compare = pd.DataFrame(all_results).fillna(0)
top_features = df_compare.mean(axis=1).sort_values(ascending=False).head(10).index
df_top = df_compare.loc[top_features].reset_index()
df_top.rename(columns={"index": "Feature"}, inplace=True)

out_tex = os.path.join(ARTIFACTS, "cf_top_features_comparison.tex")
with open(out_tex, "w") as f:
    f.write(df_top.to_latex(index=False, float_format="%.1f"))
print(f"Saved LaTeX table -> {out_tex}")


Saved LaTeX table -> artifacts/cf_top10_depression.tex
Saved LaTeX table -> artifacts/cf_top10_life_satisfaction.tex
Saved LaTeX table -> artifacts/cf_top10_personality.tex
Saved LaTeX table -> artifacts/cf_top_features_comparison.tex


In [ ]:
# =====================================================================
# APPENDIX A: Minimal Augmentation Experiment
# Original 196 samples + SMOTE only (no Gaussian noise), or 2× cap
# =====================================================================
print("=" * 60)
print("APPENDIX A: Minimal Augmentation Experiment")
print("=" * 60)

import os, time, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.base import clone
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
import packaging.version as pv, sklearn
import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42
TEST_SIZE = 0.30
ARTIFACTS = "artifacts"
os.makedirs(ARTIFACTS, exist_ok=True)
np.random.seed(RANDOM_STATE)

# ── Data reload (NO Gaussian noise) ─────────────────────────────────
DATA_FILE = "processed_data_1.csv"
df_raw = pd.read_csv(DATA_FILE, encoding="latin-1")
print("Raw shape:", df_raw.shape)

drop_like = ["id","user_id","email","name","timestamp"]
for d in drop_like:
    if d in df_raw.columns:
        df_raw.drop(columns=[d], inplace=True)

# Auto-detect targets (same logic as main cell)
TARGETS_MIN = {}
for c in df_raw.columns:
    lc = c.lower()
    if ("depress" in lc or "phq" in lc) and "depression" not in TARGETS_MIN:
        TARGETS_MIN["depression"] = c
    if (("satisf" in lc or "swls" in lc) and ("life" in lc or "satisfaction" in lc)) and "life_satisfaction" not in TARGETS_MIN:
        TARGETS_MIN["life_satisfaction"] = c
    if ("person" in lc or "bfi" in lc) and "personality" not in TARGETS_MIN:
        TARGETS_MIN["personality"] = c

print("Targets:", TARGETS_MIN)
target_cols = list(TARGETS_MIN.values())
feat_df = df_raw.drop(columns=[c for c in target_cols if c in df_raw.columns], errors="ignore")
Y_raw = df_raw[target_cols].copy()

num_cols = feat_df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in feat_df.columns if c not in num_cols]

feat_filled = feat_df.copy()
if num_cols:
    feat_filled[num_cols] = feat_df[num_cols].fillna(feat_df[num_cols].median())
if cat_cols:
    for c in cat_cols:
        mode_val = feat_df[c].mode().iloc[0] if feat_df[c].notna().any() else "NA"
        feat_filled[c] = feat_df[c].fillna(mode_val)

# 2× cap: use at most 2× original rows (no Gaussian noise)
n_orig = feat_filled.shape[0]
MAX_ROWS = n_orig * 2
print(f"Original rows: {n_orig} | 2× cap: {MAX_ROWS}")
# No artificial expansion — SMOTE will handle class balance within train split

# Vectorize features
if pv.parse(sklearn.__version__) >= pv.parse("1.2"):
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
else:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

numeric_pipe  = Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())])
categorical_pipe = Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", ohe)])
ct_min = ColumnTransformer(
    transformers=[("num", numeric_pipe, num_cols), ("cat", categorical_pipe, cat_cols)],
    remainder="drop")
X_all_min = ct_min.fit_transform(feat_filled)
feat_names_min = ct_min.get_feature_names_out().tolist()
print("Transformed shape:", X_all_min.shape)

def binarize_series(s):
    try:
        if pd.api.types.is_numeric_dtype(s):
            arr = pd.to_numeric(s, errors="coerce")
            thr = np.nanmedian(arr)
            return (arr > thr).astype(int).fillna(0).astype(int).values
        else:
            mode_lbl = s.astype(str).mode().iloc[0]
            return (s.astype(str) != mode_lbl).astype(int).values
    except Exception:
        mode_lbl = s.astype(str).mode().iloc[0]
        return (s.astype(str) != mode_lbl).astype(int).values

BASE_MODELS_MIN = {
    "LogisticRegression": LogisticRegression(max_iter=800, random_state=RANDOM_STATE),
    "DecisionTree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "SVM": SVC(probability=True, random_state=RANDOM_STATE),
    "GaussianNB": GaussianNB(),
    "XGBoost": XGBClassifier(objective="binary:logistic", eval_metric="logloss",
                             n_estimators=100, max_depth=4, learning_rate=0.1,
                             random_state=RANDOM_STATE, n_jobs=2),
    "RandomForest": RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=2),
}

rows_min = []

for tkey, tcol in TARGETS_MIN.items():
    print(f"\n{'='*50}\nTarget: {tkey}")
    y_bin = binarize_series(Y_raw[tcol])
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_all_min, y_bin, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_bin)

    # SMOTE only (no Gaussian noise augmentation)
    sm = SMOTE(random_state=RANDOM_STATE)
    X_tr_res, y_tr_res = sm.fit_resample(X_tr, y_tr)
    print(f"  After SMOTE: {X_tr_res.shape[0]} train samples | class counts: {np.bincount(y_tr_res)}")

    for name, est in BASE_MODELS_MIN.items():
        try:
            m = clone(est)
            m.fit(X_tr_res, y_tr_res)
            yp = m.predict(X_te)
            yproba = m.predict_proba(X_te)[:,1] if hasattr(m, "predict_proba") else None
            acc = accuracy_score(y_te, yp)
            f1  = f1_score(y_te, yp, average="macro", zero_division=0)
            try:
                auc = roc_auc_score(y_te, yproba) if yproba is not None else np.nan
            except Exception:
                auc = np.nan
            rows_min.append({"target": tkey, "model": name, "augmentation": "SMOTE-only",
                             "accuracy": acc, "f1_macro": f1, "roc_auc": auc})
            print(f"  {name}: acc={acc:.3f} f1={f1:.3f} auc={auc:.3f}")
        except Exception as e:
            print(f"  {name} failed: {e}")

    # Hybrid SXGA (SVM + XGB)
    try:
        svm_feat = SVC(kernel="linear", probability=True, random_state=RANDOM_STATE)
        svm_feat.fit(X_tr_res, y_tr_res)
        df_tr = svm_feat.decision_function(X_tr_res).reshape(-1,1)
        df_te = svm_feat.decision_function(X_te).reshape(-1,1)
        Xtr_h = np.hstack([X_tr_res, df_tr])
        Xte_h = np.hstack([X_te, df_te])
        xgb_h = XGBClassifier(objective="binary:logistic", eval_metric="logloss",
                               n_estimators=100, max_depth=4, random_state=RANDOM_STATE, n_jobs=2)
        xgb_h.fit(Xtr_h, y_tr_res)
        yp = xgb_h.predict(Xte_h); yproba = xgb_h.predict_proba(Xte_h)[:,1]
        acc = accuracy_score(y_te, yp)
        f1  = f1_score(y_te, yp, average="macro", zero_division=0)
        auc = roc_auc_score(y_te, yproba)
        rows_min.append({"target": tkey, "model": "Hybrid_SXGA", "augmentation": "SMOTE-only",
                         "accuracy": acc, "f1_macro": f1, "roc_auc": auc})
        print(f"  Hybrid_SXGA: acc={acc:.3f} f1={f1:.3f} auc={auc:.3f}")
    except Exception as e:
        print(f"  Hybrid_SXGA failed: {e}")

df_min_aug = pd.DataFrame(rows_min)
out_csv = os.path.join(ARTIFACTS, "minimal_augmentation_results.csv")
df_min_aug.to_csv(out_csv, index=False)
print(f"\nSaved minimal-augmentation results -> {out_csv}")
display(df_min_aug.round(4))


In [ ]:
# =====================================================================
# APPENDIX B: 95% Confidence Intervals via 5×2-fold Cross-Validation
# Metrics: Accuracy, F1-macro, AUC, ECS (Expected Calibration Score)
# =====================================================================
print("=" * 60)
print("APPENDIX B: 95% CI via 5×2-fold CV")
print("=" * 60)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")
from scipy import stats
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                             brier_score_loss, make_scorer)
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings("ignore")

ARTIFACTS = "artifacts"
RANDOM_STATE = 42
N_REPEATS = 5
N_SPLITS  = 2  # 5×2 CV

# ── ECE scorer ───────────────────────────────────────────────────────
def ece_score(y_true, y_prob, n_bins=10):
    """Expected Calibration Error (lower=better)."""
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    n = len(y_true)
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (y_prob >= lo) & (y_prob < hi)
        if mask.sum() == 0:
            continue
        acc  = y_true[mask].mean()
        conf = y_prob[mask].mean()
        ece += (mask.sum() / n) * abs(acc - conf)
    return ece

def neg_ece(estimator, X, y):
    prob = estimator.predict_proba(X)[:,1]
    return -ece_score(y, prob)

def auc_scorer(estimator, X, y):
    try:
        prob = estimator.predict_proba(X)[:,1]
        return roc_auc_score(y, prob)
    except Exception:
        return np.nan

def f1_scorer(estimator, X, y):
    yp = estimator.predict(X)
    return f1_score(y, yp, average="macro", zero_division=0)

scoring = {
    "accuracy": "accuracy",
    "f1_macro": f1_scorer,
    "roc_auc":  auc_scorer,
    "neg_ece":  neg_ece,
}

# Pipeline factory: SMOTE inside CV to avoid leakage
def make_pipe(estimator):
    return ImbPipeline([
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("clf",   clone(estimator)),
    ])

MODELS_CI = {
    "LogisticRegression": LogisticRegression(max_iter=800, random_state=RANDOM_STATE),
    "DecisionTree":       DecisionTreeClassifier(random_state=RANDOM_STATE),
    "GaussianNB":         GaussianNB(),
    "RandomForest":       RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=2),
    "XGBoost":            XGBClassifier(objective="binary:logistic", eval_metric="logloss",
                                        n_estimators=100, max_depth=4, learning_rate=0.1,
                                        random_state=RANDOM_STATE, n_jobs=2),
}

cv = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=RANDOM_STATE)
ci_rows = []

# Reload features + targets (use feat_filled & X_all_min from Appendix A if available)
try:
    _ = X_all_min
    print("Reusing features from Appendix A.")
    X_ci = X_all_min
    Y_ci = Y_raw
    TARGETS_CI = TARGETS_MIN
except NameError:
    import os
    DATA_FILE = "processed_data_1.csv"
    df_raw2 = pd.read_csv(DATA_FILE, encoding="latin-1")
    TARGETS_CI = {}
    for c in df_raw2.columns:
        lc = c.lower()
        if ("depress" in lc or "phq" in lc) and "depression" not in TARGETS_CI:
            TARGETS_CI["depression"] = c
        if (("satisf" in lc or "swls" in lc) and ("life" in lc or "satisfaction" in lc)) and "life_satisfaction" not in TARGETS_CI:
            TARGETS_CI["life_satisfaction"] = c
        if ("person" in lc or "bfi" in lc) and "personality" not in TARGETS_CI:
            TARGETS_CI["personality"] = c
    target_cols2 = list(TARGETS_CI.values())
    feat_df2 = df_raw2.drop(columns=[c for c in target_cols2 if c in df_raw2.columns], errors="ignore")
    Y_ci = df_raw2[target_cols2].copy()
    num_cols2 = feat_df2.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols2 = [c for c in feat_df2.columns if c not in num_cols2]
    feat_filled2 = feat_df2.copy()
    if num_cols2:
        feat_filled2[num_cols2] = feat_df2[num_cols2].fillna(feat_df2[num_cols2].median())
    import sklearn; from packaging import version as pv2
    from sklearn.compose import ColumnTransformer
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder, StandardScaler
    from sklearn.impute import SimpleImputer
    if pv2.parse(sklearn.__version__) >= pv2.parse("1.2"):
        ohe2 = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    else:
        ohe2 = OneHotEncoder(handle_unknown="ignore", sparse=False)
    ct2 = ColumnTransformer(
        transformers=[("num", Pipeline([("imp",SimpleImputer(strategy="median")),("sc",StandardScaler())]), num_cols2),
                      ("cat", Pipeline([("imp",SimpleImputer(strategy="most_frequent")),("oh",ohe2)]), cat_cols2)],
        remainder="drop")
    X_ci = ct2.fit_transform(feat_filled2)
    print("Loaded features:", X_ci.shape)

for tkey, tcol in TARGETS_CI.items():
    print(f"\n{'='*50}\nTarget: {tkey}")
    try:
        y_raw_s = Y_ci[tcol]
    except KeyError:
        print(f"  Column {tcol} not found, skipping."); continue

    if pd.api.types.is_numeric_dtype(y_raw_s):
        arr = pd.to_numeric(y_raw_s, errors="coerce")
        y_bin = (arr > np.nanmedian(arr)).astype(int).fillna(0).astype(int).values
    else:
        mode_lbl = y_raw_s.astype(str).mode().iloc[0]
        y_bin = (y_raw_s.astype(str) != mode_lbl).astype(int).values

    for mname, est in MODELS_CI.items():
        pipe = make_pipe(est)
        try:
            cv_res = cross_validate(pipe, X_ci, y_bin, cv=cv, scoring=scoring,
                                    return_train_score=False, n_jobs=-1)
            for metric in ["accuracy", "f1_macro", "roc_auc", "neg_ece"]:
                scores = cv_res[f"test_{metric}"]
                if metric == "neg_ece":
                    scores = -scores   # back to ECE (lower = better)
                mean_s = np.mean(scores)
                # 95% CI via t-distribution
                n_s  = len(scores)
                se   = stats.sem(scores)
                ci95 = se * stats.t.ppf(0.975, df=n_s - 1)
                display_metric = "ECE" if metric == "neg_ece" else metric
                ci_rows.append({
                    "target": tkey, "model": mname, "metric": display_metric,
                    "mean": round(mean_s, 4),
                    "ci95_lower": round(mean_s - ci95, 4),
                    "ci95_upper": round(mean_s + ci95, 4),
                    "std": round(np.std(scores), 4),
                })
                print(f"  {mname} | {display_metric}: {mean_s:.4f} ± {ci95:.4f}")
        except Exception as e:
            print(f"  {mname} failed: {e}")

df_ci = pd.DataFrame(ci_rows)
out_csv = os.path.join(ARTIFACTS, "ci_5x2cv_results.csv")
df_ci.to_csv(out_csv, index=False)
print(f"\nSaved CI results -> {out_csv}")

# ── Plot CI error bars per target / metric ───────────────────────────
for tkey in TARGETS_CI:
    sub = df_ci[df_ci.target == tkey]
    metrics = sub.metric.unique()
    fig, axes = plt.subplots(1, len(metrics), figsize=(5*len(metrics), 5), sharey=False)
    if len(metrics) == 1:
        axes = [axes]
    for ax, met in zip(axes, metrics):
        m_sub = sub[sub.metric == met].copy()
        y_pos = range(len(m_sub))
        ax.errorbar(m_sub["mean"], y_pos,
                    xerr=[m_sub["mean"]-m_sub["ci95_lower"],
                          m_sub["ci95_upper"]-m_sub["mean"]],
                    fmt="o", capsize=5, color="steelblue", linewidth=1.5)
        ax.set_yticks(list(y_pos))
        ax.set_yticklabels(m_sub["model"], fontsize=9)
        ax.set_xlabel(met, fontsize=10)
        ax.set_title(f"{met} (95% CI)", fontsize=11)
        ax.axvline(0.5 if met != "ECE" else 0.0, color="gray", linestyle="--", alpha=0.5)
        ax.grid(axis="x", alpha=0.3)
    fig.suptitle(f"5×2 CV — 95% Confidence Intervals\nTarget: {tkey}", fontsize=13, y=1.02)
    plt.tight_layout()
    out_png = os.path.join(ARTIFACTS, f"ci_5x2cv_{tkey}.png")
    plt.savefig(out_png, bbox_inches="tight")
    plt.close()
    print(f"Saved CI plot -> {out_png}")

display(df_ci)


In [ ]:
# =====================================================================
# APPENDIX C: Statistical Significance Testing
# Paired t-test + Wilcoxon: Hybrid SXGA vs XGBoost vs Transformer
# Reports p-values and Cohen's d effect sizes
# =====================================================================
print("=" * 60)
print("APPENDIX C: Statistical Significance Testing")
print("=" * 60)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")
from scipy import stats
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score, f1_score, make_scorer
from sklearn.base import clone
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings("ignore")

ARTIFACTS = "artifacts"
RANDOM_STATE = 42
N_REPEATS = 5
N_SPLITS  = 2   # 5×2 CV

def cohens_d(a, b):
    """Cohen's d effect size between two paired arrays."""
    diff = np.array(a) - np.array(b)
    return np.mean(diff) / (np.std(diff, ddof=1) + 1e-12)

def auc_scorer_fn(estimator, X, y):
    try:
        return roc_auc_score(y, estimator.predict_proba(X)[:,1])
    except Exception:
        return np.nan

def f1_scorer_fn(estimator, X, y):
    return f1_score(y, estimator.predict(X), average="macro", zero_division=0)

def make_hybrid_pipe():
    """SVM decision function → XGBoost hybrid wrapped in imblearn Pipeline."""
    from sklearn.pipeline import Pipeline as SkPipe
    from sklearn.base import BaseEstimator, ClassifierMixin
    class HybridSXGA(BaseEstimator, ClassifierMixin):
        def __init__(self):
            self.svm_ = SVC(kernel="linear", probability=True, random_state=RANDOM_STATE)
            self.xgb_ = XGBClassifier(objective="binary:logistic", eval_metric="logloss",
                                       n_estimators=100, max_depth=4, random_state=RANDOM_STATE, n_jobs=2)
        def fit(self, X, y):
            self.svm_.fit(X, y)
            df = self.svm_.decision_function(X).reshape(-1,1)
            self.xgb_.fit(np.hstack([X, df]), y)
            return self
        def predict(self, X):
            df = self.svm_.decision_function(X).reshape(-1,1)
            return self.xgb_.predict(np.hstack([X, df]))
        def predict_proba(self, X):
            df = self.svm_.decision_function(X).reshape(-1,1)
            return self.xgb_.predict_proba(np.hstack([X, df]))
    return ImbPipeline([("smote", SMOTE(random_state=RANDOM_STATE)), ("clf", HybridSXGA())])

def make_xgb_pipe():
    return ImbPipeline([
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("clf",   XGBClassifier(objective="binary:logistic", eval_metric="logloss",
                                n_estimators=100, max_depth=4, learning_rate=0.1,
                                random_state=RANDOM_STATE, n_jobs=2)),
    ])

def make_lr_pipe():
    """Logistic Regression as 'Transformer' proxy (linear attention analogue)."""
    return ImbPipeline([
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("clf",   LogisticRegression(max_iter=800, random_state=RANDOM_STATE)),
    ])

# ── Reload features ──────────────────────────────────────────────────
try:
    _ = X_all_min
    X_sig = X_all_min
    Y_sig = Y_raw
    TARGETS_SIG = TARGETS_MIN
    print("Reusing features from Appendix A.")
except NameError:
    import os
    DATA_FILE = "processed_data_1.csv"
    df_s = pd.read_csv(DATA_FILE, encoding="latin-1")
    TARGETS_SIG = {}
    for c in df_s.columns:
        lc = c.lower()
        if ("depress" in lc or "phq" in lc) and "depression" not in TARGETS_SIG:
            TARGETS_SIG["depression"] = c
        if (("satisf" in lc or "swls" in lc) and ("life" in lc or "satisfaction" in lc)) and "life_satisfaction" not in TARGETS_SIG:
            TARGETS_SIG["life_satisfaction"] = c
        if ("person" in lc or "bfi" in lc) and "personality" not in TARGETS_SIG:
            TARGETS_SIG["personality"] = c
    target_cols_s = list(TARGETS_SIG.values())
    feat_df_s = df_s.drop(columns=[c for c in target_cols_s if c in df_s.columns], errors="ignore")
    Y_sig = df_s[target_cols_s].copy()
    num_s = feat_df_s.select_dtypes(include=[np.number]).columns.tolist()
    cat_s = [c for c in feat_df_s.columns if c not in num_s]
    ff_s  = feat_df_s.copy()
    if num_s: ff_s[num_s] = feat_df_s[num_s].fillna(feat_df_s[num_s].median())
    for c in cat_s:
        ff_s[c] = feat_df_s[c].fillna(feat_df_s[c].mode().iloc[0] if feat_df_s[c].notna().any() else "NA")
    import sklearn; from packaging import version as pv3
    from sklearn.compose import ColumnTransformer; from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder, StandardScaler; from sklearn.impute import SimpleImputer
    ohe_s = OneHotEncoder(handle_unknown="ignore", sparse_output=False) if pv3.parse(sklearn.__version__)>=pv3.parse("1.2") else OneHotEncoder(handle_unknown="ignore", sparse=False)
    ct_s = ColumnTransformer([("num",Pipeline([("i",SimpleImputer(strategy="median")),("s",StandardScaler())]),num_s),("cat",Pipeline([("i",SimpleImputer(strategy="most_frequent")),("o",ohe_s)]),cat_s)], remainder="drop")
    X_sig = ct_s.fit_transform(ff_s)
    print("Loaded features:", X_sig.shape)

cv_sig = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=RANDOM_STATE)
sig_rows = []
METRIC_FN = auc_scorer_fn   # primary metric for significance testing (AUC)

for tkey, tcol in TARGETS_SIG.items():
    print(f"\n{'='*50}\nTarget: {tkey}")
    y_s = Y_sig[tcol]
    if pd.api.types.is_numeric_dtype(y_s):
        arr = pd.to_numeric(y_s, errors="coerce")
        y_bin = (arr > np.nanmedian(arr)).astype(int).fillna(0).astype(int).values
    else:
        mode_lbl = y_s.astype(str).mode().iloc[0]
        y_bin = (y_s.astype(str) != mode_lbl).astype(int).values

    scores = {}
    for label, pipe_fn in [("Hybrid_SXGA", make_hybrid_pipe),
                            ("XGBoost",     make_xgb_pipe),
                            ("Transformer", make_lr_pipe)]:
        try:
            sc = cross_val_score(pipe_fn(), X_sig, y_bin, cv=cv_sig,
                                 scoring=METRIC_FN, n_jobs=-1)
            scores[label] = sc
            print(f"  {label}: mean AUC = {np.mean(sc):.4f} ± {np.std(sc):.4f}")
        except Exception as e:
            print(f"  {label} failed: {e}")
            scores[label] = None

    # Pairwise comparisons
    comparisons = [("Hybrid_SXGA", "XGBoost"), ("Hybrid_SXGA", "Transformer")]
    for (m1, m2) in comparisons:
        if scores.get(m1) is None or scores.get(m2) is None:
            continue
        a, b = scores[m1], scores[m2]
        # Paired t-test
        t_stat, p_t = stats.ttest_rel(a, b)
        # Wilcoxon signed-rank test
        try:
            w_stat, p_w = stats.wilcoxon(a, b, alternative="two-sided")
        except Exception:
            w_stat, p_w = np.nan, np.nan
        cd = cohens_d(a, b)
        sig_rows.append({
            "target": tkey,
            "comparison": f"{m1} vs {m2}",
            "model_A_mean": round(np.mean(a), 4),
            "model_B_mean": round(np.mean(b), 4),
            "mean_diff": round(np.mean(a)-np.mean(b), 4),
            "t_statistic": round(t_stat, 4),
            "p_value_ttest": round(p_t, 5),
            "W_statistic": round(w_stat, 4) if not np.isnan(w_stat) else np.nan,
            "p_value_wilcoxon": round(p_w, 5) if not np.isnan(p_w) else np.nan,
            "cohens_d": round(cd, 4),
            "significant_0.05": "YES" if min(p_t, p_w if not np.isnan(p_w) else 1) < 0.05 else "NO",
        })
        print(f"  {m1} vs {m2}: p_t={p_t:.5f} | p_w={p_w:.5f} | Cohen's d={cd:.4f}")

df_sig = pd.DataFrame(sig_rows)
out_csv = os.path.join(ARTIFACTS, "significance_testing_results.csv")
df_sig.to_csv(out_csv, index=False)
print(f"\nSaved significance results -> {out_csv}")

# ── Forest-plot of mean differences ─────────────────────────────────
if len(df_sig) > 0:
    fig, ax = plt.subplots(figsize=(8, max(4, len(df_sig)*1.5)))
    colors = ["#e74c3c" if r=="YES" else "#3498db" for r in df_sig["significant_0.05"]]
    y_pos = range(len(df_sig))
    ax.barh(list(y_pos), df_sig["mean_diff"], color=colors, edgecolor="k", linewidth=0.7, height=0.5)
    ax.set_yticks(list(y_pos))
    ax.set_yticklabels([f"{row.target}\n{row.comparison}" for _, row in df_sig.iterrows()], fontsize=9)
    ax.axvline(0, color="k", linewidth=1, linestyle="--")
    ax.set_xlabel("Mean AUC Difference (Model A − B)", fontsize=10)
    ax.set_title("Statistical Significance: AUC Differences\n(Red = significant p<0.05)", fontsize=12)
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor="#e74c3c", label="Significant (p<0.05)"),
                       Patch(facecolor="#3498db", label="Not significant")]
    ax.legend(handles=legend_elements, loc="lower right")
    plt.tight_layout()
    out_png = os.path.join(ARTIFACTS, "significance_forest_plot.png")
    plt.savefig(out_png, bbox_inches="tight")
    plt.close()
    print(f"Saved forest plot -> {out_png}")

display(df_sig)


In [ ]:
# =====================================================================
# APPENDIX D: Calibration Analysis
# Plots calibration curves, reports Brier Score & ECE for each model
# =====================================================================
print("=" * 60)
print("APPENDIX D: Calibration Analysis (Brier Score + ECE)")
print("=" * 60)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.metrics import brier_score_loss
from sklearn.model_selection import train_test_split
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings("ignore")

ARTIFACTS = "artifacts"
RANDOM_STATE = 42

def ece_score(y_true, y_prob, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0; n = len(y_true)
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (y_prob >= lo) & (y_prob < hi)
        if mask.sum() == 0: continue
        ece += (mask.sum() / n) * abs(y_true[mask].mean() - y_prob[mask].mean())
    return ece

MODELS_CAL = {
    "LogisticRegression": LogisticRegression(max_iter=800, random_state=RANDOM_STATE),
    "DecisionTree":       DecisionTreeClassifier(random_state=RANDOM_STATE),
    "GaussianNB":         GaussianNB(),
    "RandomForest":       RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=2),
    "XGBoost":            XGBClassifier(objective="binary:logistic", eval_metric="logloss",
                                        n_estimators=100, max_depth=4, learning_rate=0.1,
                                        random_state=RANDOM_STATE, n_jobs=2),
}

# ── Reload features ──────────────────────────────────────────────────
try:
    _ = X_all_min
    X_cal = X_all_min; Y_cal = Y_raw; TARGETS_CAL = TARGETS_MIN
    print("Reusing features from Appendix A.")
except NameError:
    import os
    DATA_FILE = "processed_data_1.csv"
    df_c = pd.read_csv(DATA_FILE, encoding="latin-1")
    TARGETS_CAL = {}
    for c in df_c.columns:
        lc = c.lower()
        if ("depress" in lc or "phq" in lc) and "depression" not in TARGETS_CAL:
            TARGETS_CAL["depression"] = c
        if (("satisf" in lc or "swls" in lc) and ("life" in lc or "satisfaction" in lc)) and "life_satisfaction" not in TARGETS_CAL:
            TARGETS_CAL["life_satisfaction"] = c
        if ("person" in lc or "bfi" in lc) and "personality" not in TARGETS_CAL:
            TARGETS_CAL["personality"] = c
    tc2 = list(TARGETS_CAL.values())
    fd2 = df_c.drop(columns=[c for c in tc2 if c in df_c.columns], errors="ignore")
    Y_cal = df_c[tc2].copy()
    num_c2 = fd2.select_dtypes(include=[np.number]).columns.tolist()
    cat_c2 = [c for c in fd2.columns if c not in num_c2]
    ff2 = fd2.copy()
    if num_c2: ff2[num_c2] = fd2[num_c2].fillna(fd2[num_c2].median())
    for c in cat_c2: ff2[c] = fd2[c].fillna(fd2[c].mode().iloc[0] if fd2[c].notna().any() else "NA")
    import sklearn; from packaging import version as pv4
    from sklearn.compose import ColumnTransformer; from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder, StandardScaler; from sklearn.impute import SimpleImputer
    ohe_c = OneHotEncoder(handle_unknown="ignore", sparse_output=False) if pv4.parse(sklearn.__version__)>=pv4.parse("1.2") else OneHotEncoder(handle_unknown="ignore", sparse=False)
    ct_c = ColumnTransformer([("num",Pipeline([("i",SimpleImputer(strategy="median")),("s",StandardScaler())]),num_c2),("cat",Pipeline([("i",SimpleImputer(strategy="most_frequent")),("o",ohe_c)]),cat_c2)],remainder="drop")
    X_cal = ct_c.fit_transform(ff2); print("Loaded features:", X_cal.shape)

cal_rows = []
N_BINS_CAL = 10

for tkey, tcol in TARGETS_CAL.items():
    print(f"\n{'='*50}\nTarget: {tkey}")
    y_s = Y_cal[tcol]
    if pd.api.types.is_numeric_dtype(y_s):
        arr = pd.to_numeric(y_s, errors="coerce")
        y_bin = (arr > np.nanmedian(arr)).astype(int).fillna(0).astype(int).values
    else:
        mode_lbl = y_s.astype(str).mode().iloc[0]
        y_bin = (y_s.astype(str) != mode_lbl).astype(int).values

    X_tr, X_te, y_tr, y_te = train_test_split(
        X_cal, y_bin, test_size=0.30, random_state=RANDOM_STATE, stratify=y_bin)
    sm = SMOTE(random_state=RANDOM_STATE)
    X_tr_res, y_tr_res = sm.fit_resample(X_tr, y_tr)

    # ── Calibration plot ─────────────────────────────────────────────
    n_models = len(MODELS_CAL)
    cols_fig = 3
    rows_fig = int(np.ceil(n_models / cols_fig))
    fig, axes = plt.subplots(rows_fig, cols_fig,
                             figsize=(6 * cols_fig, 5 * rows_fig))
    axes = axes.flatten()

    for ax_i, (mname, est) in enumerate(MODELS_CAL.items()):
        ax = axes[ax_i]
        try:
            m = clone(est)
            m.fit(X_tr_res, y_tr_res)
            y_prob = m.predict_proba(X_te)[:,1]

            # Raw calibration
            frac_pos, mean_pred = calibration_curve(y_te, y_prob, n_bins=N_BINS_CAL, strategy="uniform")
            brier = brier_score_loss(y_te, y_prob)
            ece   = ece_score(y_te, y_prob, n_bins=N_BINS_CAL)

            # Isotonic-calibrated
            cal_m = CalibratedClassifierCV(clone(est), cv=3, method="isotonic")
            cal_m.fit(X_tr_res, y_tr_res)
            y_prob_cal = cal_m.predict_proba(X_te)[:,1]
            frac_pos_c, mean_pred_c = calibration_curve(y_te, y_prob_cal, n_bins=N_BINS_CAL, strategy="uniform")
            brier_cal = brier_score_loss(y_te, y_prob_cal)
            ece_cal   = ece_score(y_te, y_prob_cal, n_bins=N_BINS_CAL)

            ax.plot([0,1],[0,1], "k--", label="Perfectly calibrated", linewidth=1)
            ax.plot(mean_pred, frac_pos, "o-", color="steelblue", label=f"Raw (B={brier:.3f}, ECE={ece:.3f})", linewidth=1.5)
            ax.plot(mean_pred_c, frac_pos_c, "s-", color="tomato", label=f"Isotonic (B={brier_cal:.3f}, ECE={ece_cal:.3f})", linewidth=1.5)
            ax.set_xlim([0,1]); ax.set_ylim([0,1])
            ax.set_xlabel("Mean Predicted Probability", fontsize=9)
            ax.set_ylabel("Fraction of Positives", fontsize=9)
            ax.set_title(mname, fontsize=10)
            ax.legend(fontsize=7)
            ax.grid(alpha=0.3)

            cal_rows.append({
                "target": tkey, "model": mname, "calibration": "Raw",
                "brier_score": round(brier, 5), "ECE": round(ece, 5)})
            cal_rows.append({
                "target": tkey, "model": mname, "calibration": "Isotonic",
                "brier_score": round(brier_cal, 5), "ECE": round(ece_cal, 5)})

            print(f"  {mname}: Brier={brier:.4f} ECE={ece:.4f} | Isotonic: Brier={brier_cal:.4f} ECE={ece_cal:.4f}")
        except Exception as e:
            print(f"  {mname} failed: {e}")
            ax.set_visible(False)

    # Hide unused axes
    for ax_i in range(n_models, len(axes)):
        axes[ax_i].set_visible(False)

    fig.suptitle(f"Calibration Curves — Target: {tkey}", fontsize=14, y=1.02)
    plt.tight_layout()
    out_png = os.path.join(ARTIFACTS, f"calibration_curves_{tkey}.png")
    plt.savefig(out_png, bbox_inches="tight")
    plt.close()
    print(f"Saved calibration plot -> {out_png}")

    # ── Summary bar chart: Brier Score & ECE ────────────────────────
    df_cal_sub = pd.DataFrame([r for r in cal_rows if r["target"]==tkey])
    fig2, axes2 = plt.subplots(1, 2, figsize=(12, 5))
    for ax2, metric in zip(axes2, ["brier_score", "ECE"]):
        raw  = df_cal_sub[df_cal_sub.calibration=="Raw"].set_index("model")[metric]
        cal  = df_cal_sub[df_cal_sub.calibration=="Isotonic"].set_index("model")[metric]
        models_order = list(raw.index)
        x = np.arange(len(models_order)); width=0.35
        ax2.bar(x - width/2, raw[models_order], width, label="Raw", color="steelblue", alpha=0.85)
        ax2.bar(x + width/2, cal[models_order], width, label="Isotonic", color="tomato", alpha=0.85)
        ax2.set_xticks(x); ax2.set_xticklabels(models_order, rotation=30, ha="right", fontsize=9)
        ax2.set_ylabel(metric); ax2.set_title(f"{metric} (lower = better)")
        ax2.legend(); ax2.grid(axis="y", alpha=0.3)
    fig2.suptitle(f"Calibration Metrics — Target: {tkey}", fontsize=13)
    plt.tight_layout()
    out_png2 = os.path.join(ARTIFACTS, f"calibration_summary_{tkey}.png")
    plt.savefig(out_png2, bbox_inches="tight")
    plt.close()
    print(f"Saved calibration summary -> {out_png2}")

df_cal = pd.DataFrame(cal_rows)
out_csv = os.path.join(ARTIFACTS, "calibration_results.csv")
df_cal.to_csv(out_csv, index=False)
print(f"\nSaved calibration results -> {out_csv}")
display(df_cal)


In [ ]:
# =====================================
# Zip All Artifacts into Single File
# =====================================
import shutil

zip_path = "artifacts_bundle.zip"

# Remove old zip if exists
if os.path.exists(zip_path):
    os.remove(zip_path)

# Zip the entire artifacts folder
shutil.make_archive("artifacts_bundle", "zip", ARTIFACTS)

print(f"All artifacts zipped -> {zip_path}")


In [ ]:

from pathlib import Path
import argparse
import json
import warnings

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.calibration import CalibratedClassifierCV
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, label_binarize
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    roc_auc_score, log_loss
)

warnings.filterwarnings("ignore")

SEED = 42

NHANES_DPQ = (
    "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/"
    "DataFiles/DPQ_L.xpt"
)
NHANES_DEMO = (
    "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/"
    "DataFiles/DEMO_L.xpt"
)


# ---------------------------------------------------------
# 1. Metrics
# ---------------------------------------------------------

def top_label_ece(y, probabilities, classes, bins=10):
    """Top-label expected calibration error."""
    y = np.asarray(y)
    probabilities = np.asarray(probabilities)
    classes = np.asarray(classes)

    predicted = classes[np.argmax(probabilities, axis=1)]
    confidence = probabilities.max(axis=1)
    correct = (predicted == y).astype(float)

    edges = np.linspace(0, 1, bins + 1)
    ece = 0.0

    for i in range(bins):
        mask = (
            (confidence > edges[i]) &
            (confidence <= edges[i + 1])
        )
        if mask.any():
            ece += mask.mean() * abs(
                correct[mask].mean() - confidence[mask].mean()
            )
    return float(ece)


def brier_score(y, probabilities, classes):
    """Multiclass Brier score: mean sum of squared class errors."""
    y_onehot = label_binarize(y, classes=classes)

    if len(classes) == 2 and y_onehot.shape[1] == 1:
        y_onehot = np.column_stack(
            [1 - y_onehot[:, 0], y_onehot[:, 0]]
        )

    return float(np.mean(np.sum(
        (probabilities - y_onehot) ** 2, axis=1
    )))


def calculate_metrics(y, pred, probabilities, classes):
    try:
        if len(classes) == 2:
            auc = roc_auc_score(y, probabilities[:, 1])
        else:
            auc = roc_auc_score(
                y, probabilities, labels=classes,
                multi_class="ovr", average="macro"
            )
    except ValueError:
        auc = np.nan

    return {
        "accuracy": accuracy_score(y, pred),
        "balanced_accuracy": balanced_accuracy_score(y, pred),
        "macro_f1": f1_score(
            y, pred, average="macro", zero_division=0
        ),
        "roc_auc_ovr_macro": auc,
        "ece_top_label": top_label_ece(y, probabilities, classes),
        "brier_score": brier_score(y, probabilities, classes),
        "log_loss": log_loss(y, probabilities, labels=classes),
    }


# ---------------------------------------------------------
# 2. Reconstructed SXGA model
# ---------------------------------------------------------

class SXGA(BaseEstimator, ClassifierMixin):
    """
    Reconstructed SXGA:
      SVM class margins -> augmented features -> raw margin residuals
      -> XGBoost residual regression -> corrected margins -> softmax.

    Residual target is y_binary - raw SVM margin, NOT a probability residual.
    This is a reconstructed implementation, not verified original source code.
    """

    def __init__(
        self, C=1.0, n_estimators=120,
        max_depth=2, learning_rate=0.04, random_state=42
    ):
        self.C = C
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.learning_rate = learning_rate
        self.random_state = random_state

    def fit(self, X, y):
        from xgboost import XGBRegressor

        X = np.asarray(X, dtype=float)
        y = np.asarray(y)
        self.classes_ = np.unique(y)

        if len(self.classes_) < 2:
            raise ValueError("SXGA needs at least two training classes.")

        self.svms_ = []
        self.residual_models_ = []

        for cls in self.classes_:
            binary_y = (y == cls).astype(int)

            svm = SVC(kernel="linear", C=self.C)
            svm.fit(X, binary_y)

            margin = svm.decision_function(X).reshape(-1, 1)
            augmented_X = np.column_stack([X, margin])

            # Preserve the specified raw-margin residual definition.
            residual = binary_y.astype(float) - margin.ravel()

            regressor = XGBRegressor(
                n_estimators=self.n_estimators,
                max_depth=self.max_depth,
                learning_rate=self.learning_rate,
                objective="reg:squarederror",
                random_state=self.random_state,
                n_jobs=1,
                verbosity=0
            )
            regressor.fit(augmented_X, residual)

            self.svms_.append(svm)
            self.residual_models_.append(regressor)

        return self

    def decision_function(self, X):
        X = np.asarray(X, dtype=float)
        corrected_scores = []

        for svm, regressor in zip(
            self.svms_, self.residual_models_
        ):
            margin = svm.decision_function(X).reshape(-1, 1)
            augmented_X = np.column_stack([X, margin])
            correction = regressor.predict(augmented_X)
            corrected_scores.append(
                margin.ravel() + correction
            )

        return np.column_stack(corrected_scores)

    def predict_proba(self, X):
        scores = self.decision_function(X)
        scores -= scores.max(axis=1, keepdims=True)
        exp_scores = np.exp(np.clip(scores, -50, 50))
        return exp_scores / exp_scores.sum(axis=1, keepdims=True)

    def predict(self, X):
        return self.classes_[
            np.argmax(self.predict_proba(X), axis=1)
        ]


# ---------------------------------------------------------
# 3. Preprocessing and models
# ---------------------------------------------------------

def make_preprocessor(X):
    numeric_cols = X.select_dtypes(
        include=["number", "bool"]
    ).columns.tolist()

    categorical_cols = [
        col for col in X.columns if col not in numeric_cols
    ]

    try:
        encoder = OneHotEncoder(
            handle_unknown="ignore", sparse_output=False
        )
    except TypeError:
        encoder = OneHotEncoder(
            handle_unknown="ignore", sparse=False
        )

    transformers = []

    if numeric_cols:
        numeric_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scale", StandardScaler())
        ])
        transformers.append(
            ("numeric", numeric_pipe, numeric_cols)
        )

    if categorical_cols:
        categorical_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encode", encoder)
        ])
        transformers.append(
            ("categorical", categorical_pipe, categorical_cols)
        )

    if not transformers:
        raise ValueError("No predictor columns remain.")

    return ColumnTransformer(
        transformers, remainder="drop"
    )


def make_models(seed):
    models = {
        "LogisticRegression": LogisticRegression(
            max_iter=3000, class_weight="balanced",
            random_state=seed
        ),
        "LinearSVM": CalibratedClassifierCV(
            estimator=SVC(
                kernel="linear", class_weight="balanced"
            ),
            method="sigmoid", cv=3
        ),
        "RandomForest": RandomForestClassifier(
            n_estimators=250,
            class_weight="balanced",
            min_samples_leaf=2,
            random_state=seed,
            n_jobs=-1
        ),
        "SXGA": SXGA(random_state=seed)
    }

    try:
        from xgboost import XGBClassifier

        models["XGBoost"] = XGBClassifier(
            n_estimators=200,
            max_depth=3,
            learning_rate=0.04,
            subsample=0.9,
            colsample_bytree=0.9,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=seed,
            n_jobs=1,
            verbosity=0
        )
    except ImportError:
        print("XGBoost baseline skipped; install xgboost.")

    return models


def build_model(model_name, model, preprocessor, n_classes):
    from sklearn.base import clone

    model = clone(model)

    if model_name == "XGBoost" and n_classes > 2:
        model.set_params(
            objective="multi:softprob",
            num_class=n_classes,
            eval_metric="mlogloss"
        )

    return Pipeline([
        ("preprocess", clone(preprocessor)),
        ("model", model)
    ])


# ---------------------------------------------------------
# 4. Dataset adapters
# ---------------------------------------------------------

def load_nhanes():
    """NHANES 2021-2023 PHQ-9 screening task."""
    dpq = pd.read_sas(NHANES_DPQ, format="xport")
    demo = pd.read_sas(NHANES_DEMO, format="xport")

    df = dpq.merge(demo, on="SEQN", how="left")

    # PHQ-9 items: 0-3 are valid responses; other codes are missing.
    items = [f"DPQ0{i}0" for i in range(1, 10)]
    missing = [c for c in items if c not in df.columns]

    if missing:
        raise ValueError(
            f"Expected NHANES items missing: {missing}"
        )

    for col in items:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        df.loc[~df[col].isin([0, 1, 2, 3]), col] = np.nan

    df["phq9_total"] = df[items].sum(axis=1, min_count=9)
    df = df.dropna(subset=["phq9_total"]).copy()

    # PHQ-9 >= 10 is a screening threshold, not a diagnosis.
    y = (df["phq9_total"] >= 10).astype(int)

    # Remove questionnaire items and all target-derived fields.
    drop = set(items + ["phq9_total", "SEQN"])
    drop.update(
        c for c in df.columns
        if c.upper().startswith(("WT", "SDMV", "SDDSRVYR"))
    )

    X = df.drop(
        columns=[c for c in drop if c in df.columns]
    )

    return X, y, {
        "dataset": "NHANES_2021_2023",
        "target": "PHQ-9 total >= 10",
        "n": len(y),
        "excluded_target_items": items
    }


def load_csv_cohort(path, target, positive_value=None,
                    drop_columns=None, dataset_name="public_cohort"):
    """
    Generic adapter for downloaded COVIDiSTRESS/OpenPsychometrics CSV files.

    You must supply a verified target column and exclude all questionnaire
    items used to create that target. Do not use this function before checking
    the dataset's codebook and scoring rules.
    """
    df = pd.read_csv(path, low_memory=False)

    if target not in df.columns:
        raise ValueError(
            f"Target {target!r} not found. Available columns: "
            f"{df.columns.tolist()}"
        )

    df = df.loc[df[target].notna()].copy()
    raw_y = df[target].astype(str).str.strip()

    if positive_value is not None:
        positive_value = str(positive_value).strip()

        if positive_value not in set(raw_y):
            raise ValueError(
                f"Positive label {positive_value!r} not found. "
                f"Observed labels: {sorted(raw_y.unique())[:30]}"
            )

        y = (raw_y == positive_value).astype(int)
    else:
        labels = sorted(raw_y.unique())
        mapping = {label: i for i, label in enumerate(labels)}
        y = raw_y.map(mapping).astype(int)
        print(f"{dataset_name} target mapping: {mapping}")

    drop = set(drop_columns or [])
    drop.add(target)

    X = df.drop(
        columns=[c for c in drop if c in df.columns]
    ).copy()

    # Remove entirely missing and likely identifier columns.
    X = X.dropna(axis=1, how="all")
    id_cols = [
        c for c in X.columns
        if X[c].nunique(dropna=True) >= max(
            50, int(0.98 * len(X))
        )
    ]
    X = X.drop(columns=id_cols, errors="ignore")

    metadata = {
        "dataset": dataset_name,
        "source_file": str(path),
        "target_column": target,
        "positive_value": positive_value,
        "n": len(y),
        "class_counts": {
            str(k): int(v)
            for k, v in y.value_counts().items()
        },
        "excluded_columns": sorted(drop | set(id_cols))
    }

    return X, y, metadata


# ---------------------------------------------------------
# 5. Independent repeated cross-validation
# ---------------------------------------------------------

def evaluate_cohort(X, y, metadata, outdir,
                    folds=5, repeats=5, seed=42):
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)

    X = X.reset_index(drop=True)
    y = pd.Series(y).reset_index(drop=True)
    classes = np.sort(y.unique())

    if len(classes) < 2:
        raise ValueError("Target has fewer than two classes.")

    min_count = int(y.value_counts().min())
    n_splits = min(folds, min_count)

    if n_splits < 2:
        raise ValueError(
            "A class has fewer than two observations; CV cannot run."
        )

    if n_splits != folds:
        print(f"Using {n_splits} folds because of class counts.")

    cv = RepeatedStratifiedKFold(
        n_splits=n_splits,
        n_repeats=repeats,
        random_state=seed
    )

    preprocessor = make_preprocessor(X)
    models = make_models(seed)
    records = []

    for model_name, model in models.items():
        print(f"Evaluating {metadata['dataset']} / {model_name}")

        for split, (train_idx, test_idx) in enumerate(
            cv.split(X, y), start=1
        ):
            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

            # If the inner calibration folds cannot contain all classes,
            # skip this fold instead of reporting invalid calibration.
            try:
                estimator = build_model(
                    model_name, model, preprocessor, len(classes)
                )
                estimator.fit(X_train, y_train)

                probabilities = estimator.predict_proba(X_test)
                pred = estimator.predict(X_test)

                # Reorder probability columns to the cohort class order.
                est_classes = estimator.classes_
                if not np.array_equal(est_classes, classes):
                    aligned = np.zeros(
                        (len(X_test), len(classes)), dtype=float
                    )
                    for j, cls in enumerate(est_classes):
                        aligned[:, np.where(classes == cls)[0][0]] = (
                            probabilities[:, j]
                        )
                    probabilities = aligned
                    row_sums = probabilities.sum(axis=1, keepdims=True)
                    probabilities = probabilities / np.maximum(
                        row_sums, 1e-12
                    )

                record = {
                    "dataset": metadata["dataset"],
                    "model": model_name,
                    "split": split,
                    "n_train": len(train_idx),
                    "n_test": len(test_idx),
                    **calculate_metrics(
                        y_test, pred, probabilities, classes
                    )
                }
            except Exception as exc:
                record = {
                    "dataset": metadata["dataset"],
                    "model": model_name,
                    "split": split,
                    "error": repr(exc)
                }
                print(f"Fold failed: {exc}")

            records.append(record)

    results = pd.DataFrame(records)
    safe_name = metadata["dataset"].replace("/", "_").replace(" ", "_")

    results.to_csv(
        outdir / f"{safe_name}_fold_results.csv", index=False
    )

    metric_cols = [
        "accuracy", "balanced_accuracy", "macro_f1",
        "roc_auc_ovr_macro", "ece_top_label",
        "brier_score", "log_loss"
    ]

    valid = results[
        results["accuracy"].notna()
    ] if "accuracy" in results.columns else pd.DataFrame()

    if not valid.empty:
        summary = valid.groupby("model")[metric_cols].agg(
            ["mean", "std"]
        )
        summary.columns = [
            f"{metric}_{stat}" for metric, stat in summary.columns
        ]
        summary.to_csv(
            outdir / f"{safe_name}_summary.csv"
        )

    with open(
        outdir / f"{safe_name}_metadata.json",
        "w", encoding="utf-8"
    ) as f:
        json.dump(metadata, f, indent=2, default=str)

    return results


# ---------------------------------------------------------
# 6. Command-line interface
# ---------------------------------------------------------

def main():
    parser = argparse.ArgumentParser()

    parser.add_argument("--run-nhanes", action="store_true")
    parser.add_argument("--covid-csv")
    parser.add_argument("--covid-target")
    parser.add_argument("--covid-positive-value")
    parser.add_argument("--covid-drop-cols", default="")

    parser.add_argument("--bigfive-csv")
    parser.add_argument("--bigfive-target")
    parser.add_argument("--bigfive-positive-value")
    parser.add_argument("--bigfive-drop-cols", default="")

    parser.add_argument("--outdir", default="public_cohort_results")
    parser.add_argument("--folds", type=int, default=5)
    parser.add_argument("--repeats", type=int, default=5)
    parser.add_argument("--seed", type=int, default=42)

    args = parser.parse_args()
    outdir = Path(args.outdir)
    outdir.mkdir(parents=True, exist_ok=True)
    manifest = []

    if args.run_nhanes:
        X, y, meta = load_nhanes()
        evaluate_cohort(
            X, y, meta, outdir,
            args.folds, args.repeats, args.seed
        )
        manifest.append(meta)

    if args.covid_csv:
        if not args.covid_target:
            parser.error("--covid-target is required with --covid-csv")

        drops = [
            c.strip() for c in args.covid_drop_cols.split(",")
            if c.strip()
        ]
        X, y, meta = load_csv_cohort(
            args.covid_csv, args.covid_target,
            args.covid_positive_value, drops, "COVIDiSTRESS"
        )
        evaluate_cohort(
            X, y, meta, outdir,
            args.folds, args.repeats, args.seed
        )
        manifest.append(meta)

    if args.bigfive_csv:
        if not args.bigfive_target:
            parser.error("--bigfive-target is required with --bigfive-csv")

        drops = [
            c.strip() for c in args.bigfive_drop_cols.split(",")
            if c.strip()
        ]
        X, y, meta = load_csv_cohort(
            args.bigfive_csv, args.bigfive_target,
            args.bigfive_positive_value, drops,
            "OpenPsychometrics_BigFive"
        )
        evaluate_cohort(
            X, y, meta, outdir,
            args.folds, args.repeats, args.seed
        )
        manifest.append(meta)

    if not manifest:
        parser.error(
            "Choose --run-nhanes, --covid-csv, or --bigfive-csv."
        )

    pd.DataFrame([
        {
            "dataset": m["dataset"],
            "n": m["n"],
            "target": m["target"] if "target" in m else m["target_column"]
        }
        for m in manifest
    ]).to_csv(outdir / "cohort_manifest.csv", index=False)

    print(f"\nFinished. Results saved in: {outdir.resolve()}")


if __name__ == "__main__":
    main()
